SILVER DATA CLEANING

In [18]:
%pip install pymsql

Note: you may need to restart the kernel to use updated packages.


In [19]:
# 1. Required libraries

import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from dotenv import load_dotenv
import os

# Load environment variable

load_dotenv()

mysql_user = os.getenv("MYSQL_USER")
mysql_password = quote_plus(os.getenv("MYSQL_PASSWORD"))
mysql_host = os.getenv("MYSQL_HOST")
mysql_port = os.getenv("MYSQL_PORT")
mysql_database = os.getenv("MYSQL_DATABASE")

In [20]:
# 2. CONNECT MYSQL 

try:
    engine = create_engine(
        f"mysql+pymysql://{mysql_user}:{mysql_password}@"
        f"{mysql_host}:{mysql_port}/{mysql_database}"
        
    )

    # test connection
    with engine.connect() as connection:
        print("Successfully connected MYSQL !")


except Exception as e:
    print(f"Error: Couldn't connect to MYSQL {e}")

Successfully connected MYSQL !


In [21]:
# 3. READ BRONZE TABLE 

df = pd.read_sql(
    "SELECT * FROM bronze_sales;",
    engine
)


In [22]:
# 4. Remove exact duplicates

print("Duplicates : " , df.duplicated().sum())

df = df.drop_duplicates()

print(df.head(2))

Duplicates :  0
   row_id        order_id  order_date   ship_date     ship_mode customer_id  \
0       1  CA-2017-152156  08/11/2017  11/11/2017  Second Class    CG-12520   
1       2  CA-2017-152156  08/11/2017  11/11/2017  Second Class    CG-12520   

  customer_name   segment        country       city  ... postal_code region  \
0   Claire Gute  Consumer  United States  Henderson  ...       42420  South   
1   Claire Gute  Consumer  United States  Henderson  ...       42420  South   

        product_id   category sub_category  \
0  FUR-BO-10001798  Furniture    Bookcases   
1  FUR-CH-10000454  Furniture       Chairs   

                                        product_name   sales  \
0                  Bush Somerset Collection Bookcase  261.96   
1  Hon Deluxe Fabric Upholstered Stacking Chairs,...  731.94   

   ingestion_timestamp source_file_name                               load_id  
0  2026-09-09 18:59:01        train.csv  9baa90f0-e877-40f7-8ac8-6257a81a4d87  
1  2026-09-09 18

In [23]:
# 5. Clean text

text_columns = [
    "ship_mode",
    "customer_name",
    "segment",
    "country",
    "city",
    "state",
    "region",
    "category",
    "sub_category",
    "product_name"
]

for col in text_columns:
    df[col] = df[col].str.strip()

In [24]:
# 6. Convert Dates

df["order_date"] = pd.to_datetime(
    df["order_date"],
    format = "%d/%m/%Y",
    errors = "coerce"
)

df["ship_date"] = pd.to_datetime(
    df["ship_date"],
    format = "%d/%m/%Y",
    errors = "coerce"
)

# Converts 08/11/2017 ----> 2017-11-08

print(df.head(2))

   row_id        order_id order_date  ship_date     ship_mode customer_id  \
0       1  CA-2017-152156 2017-11-08 2017-11-11  Second Class    CG-12520   
1       2  CA-2017-152156 2017-11-08 2017-11-11  Second Class    CG-12520   

  customer_name   segment        country       city  ... postal_code region  \
0   Claire Gute  Consumer  United States  Henderson  ...       42420  South   
1   Claire Gute  Consumer  United States  Henderson  ...       42420  South   

        product_id   category sub_category  \
0  FUR-BO-10001798  Furniture    Bookcases   
1  FUR-CH-10000454  Furniture       Chairs   

                                        product_name   sales  \
0                  Bush Somerset Collection Bookcase  261.96   
1  Hon Deluxe Fabric Upholstered Stacking Chairs,...  731.94   

   ingestion_timestamp source_file_name                               load_id  
0  2026-09-09 18:59:01        train.csv  9baa90f0-e877-40f7-8ac8-6257a81a4d87  
1  2026-09-09 18:59:01        train.cs

In [25]:
# 7. Validate the dates

invalid_order_dates = df["order_date"].isna().sum()
invalid_ship_dates = df["ship_date"].isna().sum()

print("Invalid order dates: ", invalid_order_dates)
print("Inavalid_ship_dates: ", invalid_ship_dates)

# If invalid dates exist, it stops the pipeline instantly so corrupt or unparsed data never reaches your database.

assert invalid_order_dates == 0
assert invalid_ship_dates == 0

Invalid order dates:  0
Inavalid_ship_dates:  0


In [26]:
invalid_shipping = (
    df["ship_date"] < df["order_date"]
).sum()

print("Ship date before order dat:", invalid_shipping)

assert invalid_shipping == 0

Ship date before order dat: 0


In [27]:
# 8. Missing Postal Code are preserved as NULL

postal_nulls = df["postal_code"].isna().sum()

print("Missing postal codes:", postal_nulls)

Missing postal codes: 11


In [28]:
# Convert object type -> string

df["postal_code"] = df["postal_code"].astype("string")

print(df["postal_code"].dtypes)

print(df.dtypes)

string
row_id                          int64
order_id                       object
order_date             datetime64[ns]
ship_date              datetime64[ns]
ship_mode                      object
customer_id                    object
customer_name                  object
segment                        object
country                        object
city                           object
state                          object
postal_code            string[python]
region                         object
product_id                     object
category                       object
sub_category                   object
product_name                   object
sales                         float64
ingestion_timestamp    datetime64[ns]
source_file_name               object
load_id                        object
dtype: object


In [29]:
# 9. Standardize Sales

df["sales"] = df["sales"].round(2)

assert (df["sales"] > 0).all()

print(df.head(5))

   row_id        order_id order_date  ship_date       ship_mode customer_id  \
0       1  CA-2017-152156 2017-11-08 2017-11-11    Second Class    CG-12520   
1       2  CA-2017-152156 2017-11-08 2017-11-11    Second Class    CG-12520   
2       3  CA-2017-138688 2017-06-12 2017-06-16    Second Class    DV-13045   
3       4  US-2016-108966 2016-10-11 2016-10-18  Standard Class    SO-20335   
4       5  US-2016-108966 2016-10-11 2016-10-18  Standard Class    SO-20335   

     customer_name    segment        country             city  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  postal_code region       product_id         category sub_category  \
0       42420  South  FUR-BO-1000

In [30]:
# 10. Creating derived columns

df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["order_day"] = df["order_date"].dt.day

print(df.head(2))

'''for 2017-11-18 -> order_year  = 2017
                     order_month = 11
                     order_day   = 8
'''



   row_id        order_id order_date  ship_date     ship_mode customer_id  \
0       1  CA-2017-152156 2017-11-08 2017-11-11  Second Class    CG-12520   
1       2  CA-2017-152156 2017-11-08 2017-11-11  Second Class    CG-12520   

  customer_name   segment        country       city  ...   category  \
0   Claire Gute  Consumer  United States  Henderson  ...  Furniture   
1   Claire Gute  Consumer  United States  Henderson  ...  Furniture   

  sub_category                                       product_name   sales  \
0    Bookcases                  Bush Somerset Collection Bookcase  261.96   
1       Chairs  Hon Deluxe Fabric Upholstered Stacking Chairs,...  731.94   

  ingestion_timestamp source_file_name                               load_id  \
0 2026-09-09 18:59:01        train.csv  9baa90f0-e877-40f7-8ac8-6257a81a4d87   
1 2026-09-09 18:59:01        train.csv  9baa90f0-e877-40f7-8ac8-6257a81a4d87   

   order_year order_month order_day  
0        2017          11         8  
1    

'for 2017-11-18 -> order_year  = 2017\n                     order_month = 11\n                     order_day   = 8\n'

In [31]:
silver_columns = [
    "row_id",
    "order_id",
    "order_date",
    "ship_date",
    "ship_mode",
    "customer_id",
    "customer_name",
    "segment",
    "country",
    "city",
    "state",
    "postal_code",
    "region",
    "product_id",
    "category",
    "sub_category",
    "product_name",
    "sales",
    "order_year",
    "order_month",
    "order_day"
]

df = df[silver_columns]

print(df.head(2))

   row_id        order_id order_date  ship_date     ship_mode customer_id  \
0       1  CA-2017-152156 2017-11-08 2017-11-11  Second Class    CG-12520   
1       2  CA-2017-152156 2017-11-08 2017-11-11  Second Class    CG-12520   

  customer_name   segment        country       city  ... postal_code region  \
0   Claire Gute  Consumer  United States  Henderson  ...       42420  South   
1   Claire Gute  Consumer  United States  Henderson  ...       42420  South   

        product_id   category sub_category  \
0  FUR-BO-10001798  Furniture    Bookcases   
1  FUR-CH-10000454  Furniture       Chairs   

                                        product_name   sales  order_year  \
0                  Bush Somerset Collection Bookcase  261.96        2017   
1  Hon Deluxe Fabric Upholstered Stacking Chairs,...  731.94        2017   

   order_month  order_day  
0           11          8  
1           11          8  

[2 rows x 21 columns]


In [32]:
print("\n========== SILVER VALIDATION ==========")

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nInvalid dates:")
print(df["order_date"].isna().sum())
print(df["ship_date"].isna().sum())

print("\nInvalid shipping relationships:")
print((df["ship_date"] < df["order_date"]).sum())

print("\nInvalid sales:")
print((df["sales"] <= 0).sum())


========== SILVER VALIDATION ==========
Rows: 9800
Columns: 21

Missing values:
row_id            0
order_id          0
order_date        0
ship_date         0
ship_mode         0
customer_id       0
customer_name     0
segment           0
country           0
city              0
state             0
postal_code      11
region            0
product_id        0
category          0
sub_category      0
product_name      0
sales             0
order_year        0
order_month       0
order_day         0
dtype: int64

Duplicate rows:
0

Invalid dates:
0
0

Invalid shipping relationships:
0

Invalid sales:
0


In [33]:
assert df.duplicated().sum() == 0
assert df["row_id"].nunique() == len(df)
assert df["order_date"].notna().all()
assert df["ship_date"].notna().all()
assert (df["ship_date"] >= df["order_date"]).all()
assert (df["sales"] > 0).all()

print("Silver Validation Passed !")

Silver Validation Passed !


In [35]:
df.to_sql(
    name="silver_sales",
    con = engine,
    if_exists = "append",
    index = False
)

print("Silver data load successfully!")

Silver data load successfully!
